# F03B — SERAPHIM-B : La Machine à Micro-jets
> *"Découper est une prière. Chaque coupe au mauvais endroit est un blasphème."*
> — Ordre de la Rose Sacrée, Adepta Sororitas

```
╔══════════════════════════════════════════════════════════════╗
║   FRÉGATE F03B — SERAPHIM-B : LA MACHINE À MICRO-JETS       ║
║   Rôle    : Découpe chirurgicale + Assemblage + Mix final    ║
║   IN      : directives.json · voix_purifiee.wav · music.mp3  ║
║   OUT     : master_audio_mix.mp3 + final_sanctorum.mp3       ║
║   Stack   : Pydub · pyrubberband · NumPy · soundfile         ║
╚══════════════════════════════════════════════════════════════╝
```

---

## Prérequis
- **F03A terminé** : `directives.json` présent dans `F03_SERAPHIM/CODEBASE/`
- **F02 terminé** : `voix_purifiee.wav` présent dans `F02_CELESTIAN/OUT/`
- **Musique** : fichier audio dans `F03_SERAPHIM/IN/`

## Ordre des Cellules

| # | Cellule | Description |
|---|---------|-------------|
| 1 | INIT | Monter Drive, cloner SANCTORUM, vérifier les prérequis |
| 2 | INSTALLATION | Installer pydub, pyrubberband, soundfile |
| 3 | INTERFACE | Gradio — La Machine à Micro-jets |
| 4 | SR_CUSTOS | Check-in final — fleet_status = audio_ready |

---
## CELLULE 1 — INIT

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELLULE 1 — INIT                                       ║
# ╚══════════════════════════════════════════════════════════╝

import os, sys, json
from google.colab import drive

if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

DRIVE_ROOT     = '/content/drive/MyDrive/SANCTORUM'
F02_OUT        = f'{DRIVE_ROOT}/F02_CELESTIAN/OUT'
F03_IN         = f'{DRIVE_ROOT}/F03_SERAPHIM/IN'
F03_OUT        = f'{DRIVE_ROOT}/F03_SERAPHIM/OUT'
F03_CODEBASE   = f'{DRIVE_ROOT}/F03_SERAPHIM/CODEBASE'
F03_TRACKING   = f'{DRIVE_ROOT}/F03_SERAPHIM/TRACKING'
SHARED_OUT     = f'{DRIVE_ROOT}/SHARED/OUT'
LIBER_DRIVE    = f'{DRIVE_ROOT}/liber_sanctorum.json'
SANCTORUM_DIR  = '/content/SANCTORUM'

if not os.path.exists(SANCTORUM_DIR):
    !git clone https://github.com/kioka8877-ux/SANCTORUM.git {SANCTORUM_DIR} -q
else:
    !git -C {SANCTORUM_DIR} pull -q

if SANCTORUM_DIR not in sys.path:
    sys.path.insert(0, SANCTORUM_DIR)

for d in [F03_OUT, SHARED_OUT, F03_TRACKING]:
    os.makedirs(d, exist_ok=True)

# ── Vérification des prérequis ───────────────────────────────────────────
DIRECTIVES_PATH = os.path.join(F03_CODEBASE, 'directives.json')
VOICE_PATH      = os.path.join(F02_OUT, 'voix_purifiee.wav')

def check(path, label):
    status = 'OK' if os.path.exists(path) else 'MANQUANT'
    print(f'  [{status}] {label}: {path}')
    return os.path.exists(path)

fleet_status = 'unknown'
if os.path.exists(LIBER_DRIVE):
    with open(LIBER_DRIVE) as f:
        liber = json.load(f)
    fleet_status = liber.get('fleet_status', 'unknown')

print(f'[INIT] fleet_status : {fleet_status}')
print('[INIT] Vérification des prérequis :')
ok_dir  = check(DIRECTIVES_PATH, 'directives.json')
ok_vox  = check(VOICE_PATH,      'voix_purifiee.wav')

AUDIO_EXTS = {'.mp3','.wav','.flac','.ogg','.m4a'}
musics = [f for f in os.listdir(F03_IN)
          if os.path.splitext(f)[1].lower() in AUDIO_EXTS]
_music_ok = 'OK' if musics else 'MANQUANT'
print(f'  [{_music_ok}] Musique IN : {musics or "(aucune)"}')

if not all([ok_dir, ok_vox]):
    print('[INIT] ATTENTION : certains prérequis manquent.')
    print('         → Vous pourrez les uploader manuellement dans Gradio.')
else:
    print('[INIT] Tous les prérequis sont présents. Prêt.')

---
## CELLULE 2 — INSTALLATION

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELLULE 2 — INSTALLATION                               ║
# ╚══════════════════════════════════════════════════════════╝

print('[INSTALL] Dépendances F03B SERAPHIM-B...')
!pip install -q pydub>=0.25.1 soundfile numpy scipy
!pip install -q pyrubberband>=0.3.0
!pip install -q gradio>=4.31.0 matplotlib
!apt-get install -qq ffmpeg rubberband-cli 2>/dev/null
print('[INSTALL] Terminé.')

---
## CELLULE 3 — INTERFACE GRADIO — LA MACHINE À MICRO-JETS

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELLULE 3 — INTERFACE GRADIO — LA MACHINE À MICRO-JETS ║
# ║  Pipeline : directives.json → découpe → assemblage → mix ║
# ╚══════════════════════════════════════════════════════════╝

import os, sys, json, shutil, time, tempfile, hashlib
from datetime import datetime, timezone
import numpy as np
import soundfile as sf
from pydub import AudioSegment, effects as pydub_effects
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import gradio as gr

DRIVE_ROOT    = '/content/drive/MyDrive/SANCTORUM'
F02_OUT       = f'{DRIVE_ROOT}/F02_CELESTIAN/OUT'
F03_IN        = f'{DRIVE_ROOT}/F03_SERAPHIM/IN'
F03_OUT       = f'{DRIVE_ROOT}/F03_SERAPHIM/OUT'
F03_CODEBASE  = f'{DRIVE_ROOT}/F03_SERAPHIM/CODEBASE'
F03_TRACKING  = f'{DRIVE_ROOT}/F03_SERAPHIM/TRACKING'
SHARED_OUT    = f'{DRIVE_ROOT}/SHARED/OUT'
LIBER_DRIVE   = f'{DRIVE_ROOT}/liber_sanctorum.json'


# ╔══════════════════════════════════════════════════════════╗
# ║  PIPELINE MICRO-JETS                                     ║
# ╚══════════════════════════════════════════════════════════╝

def _time_stretch(audio_seg: AudioSegment, rate: float) -> AudioSegment:
    """
    Étirement temporel sans déformation de pitch via pyrubberband.
    Fallback : speed change natif pydub (affecte le pitch) si rubberband absent.
    rate > 1 → plus rapide | rate < 1 → plus lent
    """
    if abs(rate - 1.0) < 0.001:
        return audio_seg
    try:
        import pyrubberband as pyrb
        samples = np.array(audio_seg.get_array_of_samples()).astype(np.float32)
        sr      = audio_seg.frame_rate
        if audio_seg.channels == 2:
        max_val = float(1 << (8 * audio_seg.sample_width - 1))
        samples /= 32768.0
        samples /= max_val
        max_val_i = max_val  # already defined above
        stretched = np.clip(stretched * max_val_i, -max_val_i, max_val_i - 1)
        stretched = stretched.astype(np.int16 if audio_seg.sample_width == 2 else np.int32)
        if stretched.ndim == 2:
            stretched = stretched.flatten()
        return audio_seg._spawn(stretched.tobytes(),
                                overrides={'frame_rate': sr})
    except Exception:
        # Fallback pydub : speedup via frame_rate trick (pitch affecté)
        new_sr = int(audio_seg.frame_rate * rate)
        sped   = audio_seg._spawn(audio_seg.raw_data,
                                   overrides={'frame_rate': new_sr})
        return sped.set_frame_rate(audio_seg.frame_rate)


def _duck_music(music: AudioSegment, voice: AudioSegment,
                voice_offset_ms: int, ducking_db: float) -> AudioSegment:
    """
    Applique le ducking : réduit le volume de `music` de `ducking_db` dB
    pendant les parties où `voice` est active (non-silence).
    Utilise une rampe de gain (fade 30ms) pour éviter les clics.
    """
    RAMP_MS       = 30
    SILENCE_THRESH = -45  # dBFS en dessous duquel on considère silence
    duck_gain     = 10 ** (ducking_db / 20.0)  # facteur linéaire (<1)

    voice_dur = len(voice)
    music_dur = len(music)

    # Détecter les régions actives de la voix (chunks 50ms)
    CHUNK_MS = 50
    active_regions = []  # [(start_ms, end_ms), ...]
    in_active = False
    region_start = 0
    for t in range(0, voice_dur, CHUNK_MS):
        chunk = voice[t : t + CHUNK_MS]
        is_active = chunk.dBFS > SILENCE_THRESH
        if is_active and not in_active:
            in_active = True
            region_start = t
        elif not is_active and in_active:
            in_active = False
            active_regions.append((voice_offset_ms + region_start,
                                   voice_offset_ms + t))
    if in_active:
        active_regions.append((voice_offset_ms + region_start,
                               voice_offset_ms + voice_dur))

    if not active_regions:
        return music

    # Appliquer le gain réduit aux régions actives
    result = music
    for (start_ms, end_ms) in active_regions:
        s = max(0, min(start_ms - RAMP_MS, music_dur))
        e = min(end_ms + RAMP_MS, music_dur)
        if s >= e:
            continue
        segment = music[s:e]
        ducked  = segment + (20 * np.log10(duck_gain + 1e-9))
        result  = result[:s] + ducked + result[e:]

    return result


def build_music_canvas(
    music_path: str,
    directives: dict,
    crossfade_override: int | None = None,
) -> AudioSegment:
    """
    Découpe chirurgicale + assemblage selon directives.json.
    Chaque segment de la timeline est extrait, traité (speed/reverse/fade/vol)
    puis répété `loops` fois avant d'être concatené avec crossfade.
    """
    source = AudioSegment.from_file(music_path)
    cf_ms  = crossfade_override if crossfade_override is not None \
             else int(directives.get('crossfade_ms', 15))
    timeline = directives['audio_timeline']

    assembled = AudioSegment.empty()

    for i, seg in enumerate(timeline):
        start_ms = int(seg['start'] * 1000)
        end_ms   = int(seg['end']   * 1000)
        if end_ms <= start_ms:
            continue

        # Extraire le slice
        clip = source[start_ms:end_ms]

        # Reverse
        if seg.get('reverse', False):
            clip = clip.reverse()

        # Time-stretch (pitch-preserving)
        speed = float(seg.get('speed', 1.0))
        if abs(speed - 1.0) > 0.001:
            clip = _time_stretch(clip, speed)

        # Volume
        vol_pct = int(seg.get('volume_pct', 100))
        if vol_pct != 100:
            clip = clip + (20 * np.log10(vol_pct / 100.0 + 1e-9))

        # Fades
        fade_in  = int(seg.get('fade_in_ms',  0))
        fade_out = int(seg.get('fade_out_ms', 0))
        if fade_in  > 0: clip = clip.fade_in(min(fade_in,  len(clip)//2))
        if fade_out > 0: clip = clip.fade_out(min(fade_out, len(clip)//2))

        # Boucles
        loops = max(1, int(seg.get('loops', 1)))
        looped = clip
        for _ in range(loops - 1):
            if cf_ms > 0 and len(clip) > cf_ms * 2:
                looped = looped.append(clip, crossfade=cf_ms)
            else:
                looped += clip

        # Assembler avec crossfade
        if len(assembled) == 0:
            assembled = looped
        elif cf_ms > 0 and len(assembled) > cf_ms and len(looped) > cf_ms:
            assembled = assembled.append(looped, crossfade=cf_ms)
        else:
            assembled += looped

    return assembled


def mix_voice_over_music(
    music_canvas: AudioSegment,
    voice_path:   str,
    voice_offset_ms: int,
    ducking_db: float,
) -> AudioSegment:
    """
    Superpose la voix purifiée sur la musique avec ducking.
    La voix commence à voice_offset_ms dans la piste finale.
    """
    voice = AudioSegment.from_file(voice_path)

    # S'assurer que la musique est assez longue
    needed = voice_offset_ms + len(voice)
    if len(music_canvas) < needed:
        # Étendre avec silence si nécessaire
        padding = AudioSegment.silent(duration=needed - len(music_canvas),
                                      frame_rate=music_canvas.frame_rate)
        music_canvas = music_canvas + padding

    # Ducking
    if ducking_db < 0:
        music_canvas = _duck_music(music_canvas, voice, voice_offset_ms, ducking_db)

    # Overlay
    mixed = music_canvas.overlay(voice, position=voice_offset_ms)
    return mixed


def normalize_audio(seg: AudioSegment, target_dbfs: float = -3.0) -> AudioSegment:
    if seg.dBFS == float('-inf'):  # mix silencieux — évite overflow
        return seg
    delta = target_dbfs - seg.dBFS
    return seg.apply_gain(delta)


def plot_timeline(directives: dict, total_ms: int) -> str:
    """Diagramme de Gantt des segments + voix."""
    timeline = directives.get('audio_timeline', [])
    role_colors = {
        'queue': '#3498db', 'loop': '#2ecc71', 'tete': '#e74c3c',
        'drop': '#e67e22', 'bridge': '#9b59b6', 'outro': '#95a5a6',
    }

    fig, ax = plt.subplots(figsize=(14, 3.5), facecolor='#08080e')
    ax.set_facecolor('#111')

    y_pos = 0
    x_cursor = 0
    for seg in timeline:
        dur_orig = (seg['end'] - seg['start']) * 1000
        loops    = max(1, int(seg.get('loops', 1)))
        speed    = float(seg.get('speed', 1.0))
        seg_real = dur_orig * loops / max(speed, 0.1)
        color = role_colors.get(seg.get('role', 'loop'), '#888')
        ax.barh(y_pos, seg_real / 1000, left=x_cursor / 1000,
                height=0.6, color=color, alpha=0.8, edgecolor='#222')
        label = f"{seg.get('role','?')}\n×{loops}"
        ax.text(x_cursor / 1000 + seg_real / 2000, y_pos,
                label, ha='center', va='center',
                fontsize=7, color='#eee', fontfamily='monospace')
        x_cursor += seg_real

    ax.set_title('SERAPHIM-B — Timeline assemblée', color='#e74c3c',
                 fontfamily='monospace', fontsize=11)
    ax.set_xlabel('Durée (s)', color='#555', fontfamily='monospace')
    ax.set_yticks([])
    ax.tick_params(colors='#555')
    ax.spines[:].set_color('#333')

    # Légende
    handles = [plt.Rectangle((0,0),1,1, color=c, label=r)
               for r, c in role_colors.items()]
    ax.legend(handles=handles, loc='upper right',
              facecolor='#1a1a1a', edgecolor='#333',
              labelcolor='white', fontsize=8)

    plt.tight_layout()
    out = '/tmp/seraphim_b_timeline.png'
    plt.savefig(out, dpi=110, bbox_inches='tight', facecolor='#08080e')
    plt.close(fig)
    return out


def _md5(path):
    h = hashlib.md5()
    with open(path, 'rb') as fp:
        for chunk in iter(lambda: fp.read(8192), b''): h.update(chunk)
    return h.hexdigest()


def _update_liber_f03b(output_path):
    if not os.path.exists(LIBER_DRIVE): return
    with open(LIBER_DRIVE) as f:
        liber = json.load(f)
    liber['f03_seraphim']['status']      = 'done'
    liber['f03_seraphim']['output_path'] = output_path
    liber['fleet_status']                = 'audio_ready'
    liber['final_output']                = os.path.join(SHARED_OUT, 'final_sanctorum.mp3')
    liber['sr_custos']['last_validation'] = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')
    with open(LIBER_DRIVE, 'w') as f:
        json.dump(liber, f, indent=2, ensure_ascii=False)


# ── Chargement des directives ────────────────────────────────────────────
_cached_directives = {}

def load_directives_from_drive():
    global _cached_directives
    p = os.path.join(F03_CODEBASE, 'directives.json')
    if os.path.exists(p):
        with open(p) as f:
            _cached_directives = json.load(f)
        return _cached_directives, f'directives.json chargé : {len(_cached_directives.get("audio_timeline",[]))} segments | BPM {_cached_directives.get("bpm","?")}'
    return {}, '[ERREUR] directives.json absent — lancer F03A d\'abord'


def run_microjets(
    music_from_drive, music_upload,
    directives_from_drive, directives_upload,
    voice_from_drive, voice_upload,
    voice_offset_ms,
    crossfade_override,
    ducking_db,
    output_format,
    target_dbfs,
):
    status_log = []

    # ── Résoudre les 3 entrées ───────────────────────────────────────────
    # Musique
    if music_from_drive:
        cands = sorted([os.path.join(F03_IN, f) for f in os.listdir(F03_IN)
                        if os.path.splitext(f)[1].lower() in {'.mp3','.wav','.flac','.ogg','.m4a'}])
        if not cands: return None, None, '[ERREUR] Aucune musique dans F03_SERAPHIM/IN/'
        music_path = cands[0]
    elif music_upload: music_path = music_upload.name
    else: return None, None, '[ERREUR] Aucune musique sélectionnée'

    # Directives
    if directives_from_drive:
        dir_path = os.path.join(F03_CODEBASE, 'directives.json')
        if not os.path.exists(dir_path): return None, None, '[ERREUR] directives.json absent'
    elif directives_upload: dir_path = directives_upload.name
    else: return None, None, '[ERREUR] Aucun directives.json'
    with open(dir_path) as f:
        directives = json.load(f)

    # Voix
    if voice_from_drive:
        voice_path = os.path.join(F02_OUT, 'voix_purifiee.wav')
        if not os.path.exists(voice_path): voice_path = None
    elif voice_upload: voice_path = voice_upload.name
    else: voice_path = None

    status_log.append(f'[F03B] Musique     : {os.path.basename(music_path)}')
    status_log.append(f'[F03B] Directives  : {len(directives.get("audio_timeline",[]))} segments | BPM {directives.get("bpm","?")}')
    status_log.append(f'[F03B] Voix        : {os.path.basename(voice_path) if voice_path else "(aucune — musique seule)"}')
    status_log.append(f'[F03B] Ducking     : {ducking_db} dB | Offset voix : {voice_offset_ms} ms')

    t0 = time.time()
    try:
        # ── Étape 1 : Assemblage musique ─────────────────────────────────
        status_log.append('[F03B] Assemblage musique...')
        canvas = build_music_canvas(music_path, directives,
                                    crossfade_override=int(crossfade_override) if crossfade_override else None)
        status_log.append(f'[F03B] Canvas musical : {round(len(canvas)/1000, 2)}s')

        # ── Étape 2 : Mix voix ────────────────────────────────────────────
        if voice_path and os.path.exists(voice_path):
            status_log.append('[F03B] Mix voix + ducking...')
            final = mix_voice_over_music(canvas, voice_path,
                                         int(voice_offset_ms), float(ducking_db))
        else:
            status_log.append('[F03B] Pas de voix — canvas musical seul.')
            final = canvas

        # ── Étape 3 : Normalisation ───────────────────────────────────────
        final = normalize_audio(final, float(target_dbfs))

        # ── Étape 4 : Export ──────────────────────────────────────────────
        ext = output_format.lower()
        tmp_out = tempfile.NamedTemporaryFile(suffix=f'.{ext}', delete=False)
        tmp_out.close()
        export_params = {'format': ext}
        if ext == 'mp3': export_params['bitrate'] = '320k'
        final.export(tmp_out.name, **export_params)

        elapsed = round(time.time() - t0, 2)
        status_log.append(f'[F03B] Export terminé en {elapsed}s')

        # ── Drive : master_audio_mix + final_sanctorum ───────────────────
        out_master  = os.path.join(F03_OUT,    f'master_audio_mix.{ext}')
        out_sanctum = os.path.join(SHARED_OUT, f'final_sanctorum.{ext}')
        shutil.copy(tmp_out.name, out_master)
        shutil.copy(tmp_out.name, out_sanctum)
        status_log.append(f'[F03B] Drive F03  : {out_master}')
        status_log.append(f'[F03B] SHARED/OUT : {out_sanctum}')
        status_log.append(f'[F03B] MD5        : {_md5(out_master)}')

        # ── Diagramme timeline ────────────────────────────────────────────
        timeline_png = plot_timeline(directives, len(final))

        # ── Liber + log ───────────────────────────────────────────────────
        _update_liber_f03b(out_master)
        status_log.append('[F03B] liber_sanctorum.json → audio_ready')

        log_path = os.path.join(F03_TRACKING, 'F03_LOG.md')
        ts = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')
        with open(log_path, 'a') as f:
            f.write(f'\n## [{ts}] MIX FINAL\n'
                    f'dur={round(len(final)/1000,2)}s | elapsed={elapsed}s | '
                    f'ducking={ducking_db}dB | offset_voix={voice_offset_ms}ms\n')

        status_log.append('[F03B] ═══════════════════════════════════')
        status_log.append('[F03B] MISSION ACCOMPLIE — FLOTTE COMPLETE')
        status_log.append('[F03B] ═══════════════════════════════════')

        return tmp_out.name, timeline_png, '\n'.join(status_log)

    except Exception as e:
        import traceback
        status_log.append(f'[ERREUR] {e}\n{traceback.format_exc()}')
        return None, None, '\n'.join(status_log)


# ╔══════════════════════════════════════════════════════════╗
# ║  INTERFACE GRADIO                                        ║
# ╚══════════════════════════════════════════════════════════╝

CSS = """
.gradio-container { background: #06060c; }
#seraphim-b-header { text-align:center; padding:16px; border:1px solid #3a1a0a;
    background: linear-gradient(135deg, #1a0800 0%, #06060c 100%); margin-bottom:12px; }
#seraphim-b-header h1 { color: #e74c3c; font-family:monospace; font-size:1.4em; }
#seraphim-b-header p  { color:#666; font-size:0.85em; font-family:monospace; }
button.primary { background: #3a1a0a !important; color: #f5b8b8 !important; }
"""

with gr.Blocks(css=CSS, title='F03B SERAPHIM-B — La Machine à Micro-jets') as demo:

    gr.HTML("""
    <div id='seraphim-b-header'>
      <h1>F03B — SERAPHIM-B : La Machine à Micro-jets</h1>
      <p>Découpe chirurgicale · Assemblage · Mix final · master_audio_mix.mp3</p>
      <p>Ordre de la Rose Sacrée — Adepta Sororitas</p>
    </div>
    """)

    with gr.Tabs():

        # ── Onglet 1 : Assemblage ─────────────────────────────────────
        with gr.Tab('Assemblage & Mix'):

            gr.Markdown('### Sources')
            with gr.Row():
                with gr.Column():
                    gr.Markdown('**Musique trend**')
                    music_from_drive = gr.Checkbox(label='Depuis Drive (F03_SERAPHIM/IN/)', value=True)
                    music_upload     = gr.File(label='Ou uploader', file_types=['.mp3','.wav','.flac'])

                with gr.Column():
                    gr.Markdown('**directives.json**')
                    dir_from_drive   = gr.Checkbox(label='Depuis Drive (F03_SERAPHIM/CODEBASE/)', value=True)
                    dir_upload       = gr.File(label='Ou uploader', file_types=['.json'])
                    load_dir_btn     = gr.Button('Charger et inspecter', size='sm')
                    dir_info         = gr.Textbox(label='', lines=2, interactive=False, show_label=False)

                with gr.Column():
                    gr.Markdown('**Voix purifiée (F02)**')
                    voice_from_drive = gr.Checkbox(label='Depuis Drive (F02_CELESTIAN/OUT/)', value=True)
                    voice_upload     = gr.File(label='Ou uploader', file_types=['.wav','.mp3','.flac'])

            gr.Markdown('### Paramètres de mixage')
            with gr.Row():
                voice_offset = gr.Slider(
                    label='Offset voix (ms) — quand la voix entre dans le mix',
                    minimum=0, maximum=10000, step=100, value=0
                )
                crossfade_ov = gr.Slider(
                    label='Crossfade override (ms, 0 = lire depuis directives)',
                    minimum=0, maximum=200, step=5, value=0
                )
                ducking      = gr.Slider(
                    label='Ducking musique sous voix (dB)',
                    minimum=-24, maximum=0, step=1, value=-14
                )

            with gr.Row():
                out_format = gr.Dropdown(
                    label='Format de sortie', choices=['mp3','wav','flac'], value='mp3'
                )
                target_dbfs = gr.Slider(
                    label='Normalisation cible (dBFS)',
                    minimum=-12, maximum=-1, step=0.5, value=-3.0
                )

            run_btn = gr.Button('LANCER LES MICRO-JETS', variant='primary', size='lg')

            with gr.Row():
                audio_out   = gr.Audio(
                    label='master_audio_mix (F03_SERAPHIM/OUT/ + SHARED/OUT/)',
                    type='filepath', scale=3
                )
                status_out  = gr.Textbox(label='Journal de mission', lines=14, interactive=False, scale=2)

        # ── Onglet 2 : Timeline Gantt ──────────────────────────────────
        with gr.Tab('Timeline assemblée'):
            gr.Markdown('### Diagramme Gantt des segments')
            timeline_img = gr.Image(label='Timeline', type='filepath')

        # ── Onglet 3 : Directives JSON ─────────────────────────────────
        with gr.Tab('Directives JSON'):
            gr.Markdown('### directives.json actives')
            dir_json_out  = gr.JSON(label='directives.json')
            refresh_dir   = gr.Button('Recharger depuis Drive')

        # ── Onglet 4 : Statut Flotte ───────────────────────────────────
        with gr.Tab('Statut Flotte'):
            liber_out   = gr.JSON(label='liber_sanctorum.json')
            refresh_lib = gr.Button('Lire le Liber')

    # ── Câblage ──────────────────────────────────────────────────────────

    load_dir_btn.click(
        load_directives_from_drive,
        inputs=[],
        outputs=[dir_json_out, dir_info]
    )

    run_btn.click(
        run_microjets,
        inputs=[
            music_from_drive, music_upload,
            dir_from_drive, dir_upload,
            voice_from_drive, voice_upload,
            voice_offset, crossfade_ov, ducking,
            out_format, target_dbfs,
        ],
        outputs=[audio_out, timeline_img, status_out]
    )

    refresh_dir.click(
        load_directives_from_drive,
        inputs=[],
        outputs=[dir_json_out, dir_info]
    )

    refresh_lib.click(
        lambda: json.load(open(LIBER_DRIVE)) if os.path.exists(LIBER_DRIVE) else {},
        inputs=[],
        outputs=[liber_out]
    )

print('[GRADIO] Démarrage interface La Machine à Micro-jets...')
demo.launch(share=True, debug=True, server_port=7863, inbrowser=False)

---
## CELLULE 4 — SR_CUSTOS CHECK-IN FINAL

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELLULE 4 — SR_CUSTOS CHECK-IN F03B — MISSION FINALE   ║
# ╚══════════════════════════════════════════════════════════╝

import os, sys, json, shutil

DRIVE_ROOT    = '/content/drive/MyDrive/SANCTORUM'
SANCTORUM_DIR = '/content/SANCTORUM'
LIBER_DRIVE   = f'{DRIVE_ROOT}/liber_sanctorum.json'
OUT_PATH      = f'{DRIVE_ROOT}/F03_SERAPHIM/OUT/master_audio_mix.mp3'

# Essayer aussi .wav si .mp3 absent
if not os.path.exists(OUT_PATH):
    OUT_PATH = OUT_PATH.replace('.mp3', '.wav')

if not os.path.exists(OUT_PATH):
    print('[SR_CUSTOS] ERREUR: master_audio_mix absent. Lancer la cellule 3 d\'abord.')
else:
    custos = os.path.join(SANCTORUM_DIR, 'SR_CUSTOS.py')
    if os.path.exists(custos):
        local_liber = os.path.join(SANCTORUM_DIR, 'liber_sanctorum.json')
        shutil.copy(LIBER_DRIVE, local_liber)
        !python {custos} --mode check-in --frigate F03B --output {OUT_PATH}
        shutil.copy(local_liber, LIBER_DRIVE)

# ── Sync logs CUSTOS vers Drive (M-04) ─────────────────────────────────
for _log_name in ['SR_CAMPAIGN_LOG.md', 'SR_TRANSFER_LOG.md']:
    _log_src = os.path.join(SANCTORUM_DIR, 'TRACKING', _log_name)
    if os.path.exists(_log_src):
        shutil.copy(_log_src, os.path.join(DRIVE_ROOT, 'TRACKING', _log_name))
    else:
        with open(LIBER_DRIVE) as f: liber = json.load(f)
        liber['f03_seraphim']['status']      = 'done'
        liber['f03_seraphim']['output_path'] = OUT_PATH
        liber['fleet_status']                = 'audio_ready'
        with open(LIBER_DRIVE, 'w') as f: json.dump(liber, f, indent=2, ensure_ascii=False)

    with open(LIBER_DRIVE) as f:
        liber = json.load(f)
    print('\n╔══════════════════════════════════════════════════════╗')
    print('║          ÉTAT FINAL DE LA FLOTTE SANCTORUM            ║')
    print('╠══════════════════════════════════════════════════════╣')
    print(f"║  fleet_status : {liber.get('fleet_status','n/a'):<36}║")
    print(f"║  F01 DOMINION : {liber['f01_dominion']['status']:<36}║")
    print(f"║  F02 CELESTIAN: {liber['f02_celestian']['status']:<36}║")
    print(f"║  F03 SERAPHIM : {liber['f03_seraphim']['status']:<36}║")
    final_out = liber.get('final_output', '')
    print(f"║  final_output : {os.path.basename(final_out):<36}║")
    print('╠══════════════════════════════════════════════════════╣')
    print('║  FLOTTE SANCTORUM — MISSION ACCOMPLIE                ║')
    print('║  final_sanctorum.mp3 → SHARED/OUT/                  ║')
    print('╚══════════════════════════════════════════════════════╝')